<a href="https://colab.research.google.com/github/zune2/colab-notebooks/blob/main/2026_10_05_EP005_LangChain_RAG_FakeListLLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain_huggingface langchain_community chromadb

In [2]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms.fake import FakeListLLM
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ==========================================
# 1. 가짜 데이터베이스(Vector Store) 구축
# ==========================================
# 테스트용 가짜 지식 데이터 문서들
test_docs = [
    Document(page_content="LangChain은 LLM 애플리케이션 개발을 위한 프레임워크입니다.", metadata={"source": "doc1"}),
    Document(page_content="RAG는 검색 증강 생성(Retrieval-Augmented Generation)의 약자입니다.", metadata={"source": "doc2"}),
    Document(page_content="텍스트 분할기는 긴 문서를 조각내어 임베딩 효율을 높입니다.", metadata={"source": "doc3"})
]

# 가벼운 무료 로컬 임베딩 모델 로드
embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")

# 로컬 메모리형 벡터 저장소 생성 및 검색기(Retriever) 설정
vector_store = Chroma.from_documents(test_docs, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 1}) # 가장 유사한 문서 1개만 추출


# ==========================================
# 2. FakeListLLM 정의 (검증용 가짜 LLM)
# ==========================================
# LLM 단계에서 반환할 가짜 답변들을 순서대로 리스트에 정의합니다.
# invoke가 호출될 때마다 이 리스트의 원소들이 차례대로 출력됩니다.
fake_responses = [
    "🤖 [Fake LLM 1차 답변]: 파이프라인 검증 성공! 체인이 정상 동작합니다."
]
llm = FakeListLLM(responses=fake_responses)


# ==========================================
# 3. 데이터 흐름을 추적하기 위한 중간 검증용 함수
# ==========================================
# 이 함수는 파이프라인 중간에서 검색기(Retriever)가 어떤 문맥을 찾아왔는지 콘솔에 찍어줍니다.
def debug_context(context_text):
    print("\n🔍 [RAG 시스템이 검색해온 문맥(Context)]")
    print(f"👉 \"{context_text.strip()}\"")
    print("-" * 50)
    return context_text


# ==========================================
# 4. RAG 프롬프트 및 체인(LCEL) 조립
# ==========================================
# 구조만 테스트할 목적이므로 간단한 형태로 구성합니다.
raw_prompt = """아래 문맥을 참고하여 질문에 답하세요.
문맥: {context}
질문: {question}
정답:"""

prompt = PromptTemplate.from_template(raw_prompt)

# 문서를 텍스트로 합쳐주면서 디버깅용 출력을 함께 실행하는 파이프라인 함수
def format_and_debug_docs(docs):
    context_text = "\n".join(doc.page_content for doc in docs)
    return debug_context(context_text)

# LangChain Expression Language (LCEL) 체인 연결
rag_chain = (
    {"context": retriever | format_and_debug_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


# ==========================================
# 5. 아키텍처 및 동작 검증 실행
# ==========================================
query = "RAG가 무슨 뜻이야?"
print(f"[사용자 질문 던지기]: {query}")

# 체인 작동
final_result = rag_chain.invoke(query)

print(f"[최종 체인 출력 결과]:\n{final_result}")


/tmp/ipykernel_5568/1984430821.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[사용자 질문 던지기]: RAG가 무슨 뜻이야?

🔍 [RAG 시스템이 검색해온 문맥(Context)]
👉 "RAG는 검색 증강 생성(Retrieval-Augmented Generation)의 약자입니다."
--------------------------------------------------
[최종 체인 출력 결과]:
🤖 [Fake LLM 1차 답변]: 파이프라인 검증 성공! 체인이 정상 동작합니다.


FakeListLLM을 이용하면 로컬 LLM이나 유료 API 비용 걱정 없이, 오직 랭체인(LangChain)과 RAG의 핵심 파이프라인(문서 검색 및 전달)이 구조적으로 잘 맞물려 돌아가는지 100% 확실하게 검증할 수 있습니다.
이 방식은 LLM의 답변 생성(Generation) 단계를 배제하고, "질문에 맞는 올바른 문맥(Context)을 뽑아와서 프롬프트에 정상적으로 주입하는가"를 테스트하는 데 가장 이상적인 구조입니다.

------------------------------

## 🛠️ 사전 설치
가벼운 로컬 벡터 저장소인 Chroma와 오픈소스 임베딩 라이브러리만 설치하면 됩니다.

```
pip install langchain-core langchain-community langchain-huggingface chromadb
```

## 🎯 실행 결과 예시 (검증 성공 시)

코드를 실행하면 다음과 같이 콘솔에 찍히게 됩니다.
```
[사용자 질문 던지기]: RAG가 무슨 뜻이야?

🔍 [RAG 시스템이 검색해온 문맥(Context)]
👉 "RAG는 검색 증강 생성(Retrieval-Augmented Generation)의 약자입니다."
```
```
[최종 체인 출력 결과]:
🤖 [Fake LLM 1차 답변]: 파이프라인 검증 성공! 체인이 정상 동작합니다.
```

## 💡 이 구조가 지니는 의의 (무엇을 검증했는가?)

   1. 지식 검색(Retrieval) 정상 작동 검증: 질문이 "RAG가 무슨 뜻이야?"였을 때, 벡터 데이터베이스가 test_docs 중 정확히 2번째 RAG 관련 문장을 찾아내어 debug_context에 밀어 넣었음을 확인할 수 있습니다.
   2. LCEL 파이프라인 흐름(Flow) 검증: 검색기 ➡️ 프롬프트 ➡️ LLM ➡️ 아웃풋 파서까지 데이터가 단 한 곳도 끊기지 않고 하나의 유기적인 파이프라인(|)으로 연결되어 종착지까지 무사히 도달함을 증명했습니다.
   3. 환경 격리: LLM의 성능 부족으로 인한 비정상 출력(문장 끊김, 환각) 변수를 완벽히 통제한 채, 오직 LangChain 프레임워크의 설계 자체만 완벽히 테스트할 수 있습니다.

이 검증 구조가 성공적으로 작동했다면, 다음 단계로 확장하기 위해 어떤 작업을 해보고 싶으신가요?

* 실제 하드디스크 폴더에 벡터 DB 데이터를 영구히 저장하고 로드하는 코드 추가하기
* 가짜 데이터(test_docs) 대신 내 컴퓨터 속 실제 텍스트(.txt) 파일을 읽어오는 로더 적용하기
* Fake LLM을 걷어내고, 성능이 검증된 외부 무료/유료 LLM API(예: OpenAI, Anthropic 등)로 교체해보기


성공적으로 파이프라인 검증이 완료되었습니다! 🎉
출력 결과를 보면 우리가 의도한 LangChain과 RAG의 핵심 설계 구조가 완벽하게 맞물려 작동함을 증명하고 있습니다.

## 📊 이번 검증 결과의 핵심 체크포인트

   1. 정확한 맥락 캐치 (Retrieval 성공):
   사용자가 "RAG가 무슨 뜻이야?"라고 질문했을 때, 벡터 데이터베이스(Chroma)가 세 개의 문서 중 정확히 관련 지식이 담긴 "RAG는 검색 증강 생성..." 문장을 찾아내어 프롬프트에 주입했습니다.
   2. 데이터 파이프라인 완성 (LCEL 성공):
   질문 입력 ➡️ 문서 검색 ➡️ 프롬프트 조립 ➡️ LLM 전달 ➡️ 출력 파서까지 데이터가 중간에 끊기거나 누수되지 않고 하나의 유기적인 체인(|)으로 매끄럽게 흘러갔음을 의미합니다.
   3. 환경 격리 테스트 완성:
   초경량 로컬 LLM을 썼을 때 발생했던 문장 끊김이나 환각 현상(AI 개발, 시각화 등 엉뚱한 답변)을 완전히 배제하고, 오직 LangChain의 시스템 설계가 올바른가만 깔끔하게 검증해 냈습니다.

------------------------------
## 🚀 다음 단계로 도약하기

랭체인의 뼈대를 확실하게 검증하셨으니, 이제 이 구조 위에 실제 유용한 기능들을 하나씩 살을 붙여 확장할 수 있습니다. 어떤 실습을 가장 먼저 시도해 보고 싶으신가요?

* 🗂️ 실제 데이터 적용 (Document Loader): 코드에 직접 적어둔 가짜 텍스트 리스트 대신, 내 컴퓨터에 있는 실제 메모장 파일(.txt)이나 PDF 파일을 자동으로 읽어와서 저장하는 구조 만들기

* 💾 DB 영구 저장 (Persistent Vector Store): 프로그램이 종료되면 지워지는 메모리 저장 방식이 아니라, 내 컴퓨터 특정 로컬 폴더에 벡터 데이터를 영구히 저장하고 언제든 불러와서 재사용하는 방법 알아보기

* 🔌 진짜 스마트한 LLM 연동 (Production API): 구조 검증이 끝났으니 FakeListLLM 자리에 실제 똑똑한 답변을 생성해 주는 외부 LLM API(무료 크레딧 범위 내 혹은 고성능 오픈소스 API)를 연결해 진짜 RAG 답변 확인하기

## 향후 할 일
* 내 컴퓨터의 실제 파일(PDF/TXT) 읽어오기
* 하드디스크에 벡터 데이터 영구 저장하기
* Fake LLM을 걷어내고 진짜 외부 LLM API 연결하기


